# 🚀 FREE Colab Training - Gurukul Lite Enhanced

## Optimize Accuracy: 66.7% → 80-85%

### 🎯 Goals:
- Fix English mixing in Indian languages
- Improve overall generation quality
- Train model with language-locking instructions
- Complete in 4-6 hours (FREE Colab T4 GPU)

### ⚡ Optimizations for FREE Colab:
- ✅ 2 epochs (not 3) → Faster completion
- ✅ Smaller batches → Less memory
- ✅ Frequent checkpoints → Resume if disconnected
- ✅ Reduced dataset size → Under 12-hour limit

### ⏱️ Timeline:
- Upload data: 10 minutes
- Training: 4-6 hours
- Total: ~5-6 hours

### 💰 Cost: **$0** (100% FREE!)


## 📦 Step 1: Check GPU & Install Packages

First, let's verify you have a GPU and install required packages.


In [ ]:
# Check GPU availability
!nvidia-smi -L

import torch
print(f"\n✅ PyTorch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU! Go to Runtime → Change runtime type → Select T4 GPU")


In [ ]:
# Install required packages (takes 2-3 minutes)
print("📦 Installing packages...")

!pip install -q transformers==4.35.0
!pip install -q peft==0.6.0
!pip install -q datasets==2.14.0
!pip install -q accelerate==0.24.0
!pip install -q bitsandbytes==0.41.0

print("✅ All packages installed!")


## 📁 Step 2: Upload Training Data

**⚠️ IMPORTANT:** You need to upload your training data files!

### Method 1: Google Drive (Recommended - Faster)
1. Upload your `data/` folder to Google Drive
2. Run the cell below to mount Drive

### Method 2: Direct Upload
1. Use Files panel (📁 icon on left)
2. Create folders: `data/training` and `data/validation`
3. Upload all .txt files (takes 10 minutes for 2.5GB)


In [ ]:
# OPTION A: Mount Google Drive (if you uploaded data there)
from google.colab import drive
drive.mount('/content/drive')

# Set paths (CHANGE THIS to match your Drive structure)
TRAINING_DIR = '/content/drive/MyDrive/Project/data/training'
VALIDATION_DIR = '/content/drive/MyDrive/Project/data/validation'

print(f"✅ Google Drive mounted!")
print(f"   Training: {TRAINING_DIR}")
print(f"   Validation: {VALIDATION_DIR}")


In [ ]:
# OPTION B: Direct upload (if using Files panel)
import os

os.makedirs('data/training', exist_ok=True)
os.makedirs('data/validation', exist_ok=True)

TRAINING_DIR = 'data/training'
VALIDATION_DIR = 'data/validation'

print("📁 Directories created!")
print("   Now upload .txt files using Files panel on the left")
print("   Upload to data/training/ and data/validation/")
print("\n⚠️ Comment out this cell and uncomment the Drive cell above if using Google Drive")


## 🤖 Step 3: Load Base Model

Loading BLOOMZ-560m with 8-bit quantization for efficient training.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

MODEL_NAME = "bigscience/bloomz-560m"
OUTPUT_DIR = "gurukul_lite_enhanced"

print(f"🤖 Loading {MODEL_NAME}...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("✅ Tokenizer loaded")

# Load model with 8-bit quantization
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    load_in_8bit=True,
    device_map="auto",
    torch_dtype=torch.float16
)

model = prepare_model_for_kbit_training(model)

print(f"✅ Model loaded on {torch.cuda.get_device_name(0)}")
print(f"   Memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
# Configure LoRA for parameter-efficient fine-tuning
lora_config = LoraConfig(
    r=16,  # Rank
    lora_alpha=32,  # Scaling factor
    target_modules=["query_key_value"],  # BLOOM attention modules
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("\n✅ LoRA configured - only training 0.6% of parameters!")


## 📊 Step 4: Load & Prepare Training Data

Loading data with language-locking prompts for better accuracy!


In [ ]:
import glob
from datasets import Dataset

# Language-locking prefixes - KEY IMPROVEMENT!
LANGUAGE_PREFIXES = {
    'hi': 'केवल हिंदी में लिखें:',
    'en': 'Write in English only:',
    'ta': 'தமிழில் மட்டும் எழுதுங்கள்:',
    'te': 'తెలుగులో మాత్రమే రాయండి:',
    'bn': 'শুধুমাত্র বাংলায় লিখুন:',
    'mr': 'फक्त मराठीत लिहा:',
    'gu': 'ફક્ત ગુજરાતીમાં લખો:',
    'kn': 'ಕನ್ನಡದಲ್ಲಿ ಮಾತ್ರ ಬರೆಯಿರಿ:',
    'ml': 'മലയാളത്തിൽ മാത്രം എഴുതുക:',
    'pa': 'ਸਿਰਫ਼ ਪੰਜਾਬੀ ਵਿੱਚ ਲਿਖੋ:',
    'sa': 'केवलं संस्कृते लिखतु:',
    'ur': 'صرف اردو میں لکھیں:',
    'or': 'କେବଳ ଓଡ଼ିଆରେ ଲେଖନ୍ତୁ:',
    'as': 'কেৱল অসমীয়াত লিখক:',
    'ne': 'नेपालीमा मात्र लेख्नुहोस्:',
}

def load_and_prepare_data(directory, max_per_file=500):
    """Load data with language-locking prompts"""
    texts = []
    
    for filepath in sorted(glob.glob(f"{directory}/*.txt")):
        filename = filepath.split('/')[-1]
        lang_code = filename.split('_')[0]  # Extract language code
        prefix = LANGUAGE_PREFIXES.get(lang_code, '')
        
        with open(filepath, 'r', encoding='utf-8') as f:
            lines = [line.strip() for line in f.readlines()[:max_per_file] 
                    if line.strip() and len(line.strip()) > 10]
            
            # Add language-locking prefix to each example
            for line in lines:
                if prefix:
                    texts.append(f"{prefix} {line}")
                else:
                    texts.append(line)
        
        print(f"   {filename}: {len(lines)} samples (with language-locking)")
    
    return texts

print("📊 Loading training data...")
train_texts = load_and_prepare_data(TRAINING_DIR, max_per_file=500)

print("\n📊 Loading validation data...")
val_texts = load_and_prepare_data(VALIDATION_DIR, max_per_file=100)

print(f"\n✅ Training: {len(train_texts)} samples")
print(f"✅ Validation: {len(val_texts)} samples")


In [ ]:
# Tokenize datasets (takes 5-10 minutes)
print("🔧 Tokenizing datasets...")

train_dataset = Dataset.from_dict({'text': train_texts})
val_dataset = Dataset.from_dict({'text': val_texts})

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])
tokenized_val = val_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

print("✅ Tokenization complete!")


## ⚙️ Step 5: Configure Training

Optimized for FREE Colab's 12-hour limit!


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Training configuration - OPTIMIZED FOR FREE COLAB!
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,  # ⚡ 2 epochs instead of 3
    per_device_train_batch_size=2,  # ⚡ Small batch for memory
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  # Effective batch = 2*8 = 16
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=25,
    eval_steps=100,
    save_steps=250,  # ⚡ FREQUENT checkpoints!
    save_total_limit=2,
    evaluation_strategy="steps",
    save_strategy="steps",
    load_best_model_at_end=True,
    fp16=True,
    optim="adamw_torch",
    report_to="none",
    push_to_hub=False
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

total_steps = len(tokenized_train) // (2 * 8) * 2
print(f"✅ Training configured!")
print(f"   Total steps: ~{total_steps}")
print(f"   Estimated time: {total_steps * 1.5 / 3600:.1f} - {total_steps * 2.5 / 3600:.1f} hours")
print(f"   ⚠️ Should finish in 4-6 hours ✅")


## 🚀 Step 6: START TRAINING!

**⚠️ IMPORTANT:**
- This will take **4-6 hours**
- **DO NOT close this tab!**
- Keep browser active (play a YouTube video in another tab)
- Training auto-saves every 250 steps
- If disconnected, re-run this cell to resume!


In [ ]:
import time

print("="*80)
print("  🚀 STARTING TRAINING!")
print("="*80)
print(f"\n⏰ Estimated time: 4-6 hours")
print("⚠️ Keep this tab open and active!\n")

start_time = time.time()

# Check for existing checkpoint (for resume)
import glob as checkpoint_glob
checkpoints = checkpoint_glob.glob(f"{OUTPUT_DIR}/checkpoint-*")

if checkpoints:
    latest = sorted(checkpoints)[-1]
    print(f"📂 Found checkpoint: {latest}")
    print("   Resuming from checkpoint...\n")
    trainer.train(resume_from_checkpoint=latest)
else:
    print("Starting fresh training...\n")
    trainer.train()

elapsed = time.time() - start_time
print(f"\n✅ TRAINING COMPLETE in {elapsed/3600:.2f} hours!")


## 💾 Step 7: Save & Download Model


In [ ]:
# Save the fine-tuned model
print("💾 Saving model...")

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"✅ Model saved to {OUTPUT_DIR}/")

# Create zip for download
print("\n📦 Creating zip file...")
!zip -r gurukul_lite_enhanced.zip {OUTPUT_DIR}

import os
zip_size = os.path.getsize('gurukul_lite_enhanced.zip') / 1e6
print(f"✅ Zip created: gurukul_lite_enhanced.zip ({zip_size:.1f} MB)")
print("\n📥 Download this file using Files panel on the left!")


## 🧪 Step 8: Test the New Model

Let's test if language-locking worked!


In [ ]:
# Test generation with language-locking
def test_gen(prompt, max_new=80):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new,
        temperature=0.6,
        top_p=0.85,
        top_k=40,
        do_sample=True,
        repetition_penalty=1.3
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("="*80)
print("  TESTING NEW MODEL")
print("="*80)

# Test the problematic languages
tests = [
    ("Hindi", "केवल हिंदी में लिखें: महात्मा गांधी"),
    ("Tamil", "தமிழில் மட்டும் எழுதுங்கள்: தமிழ் இலக்கியம்"),
    ("Bengali", "শুধুমাত্র বাংলায় লিখুন: রবীন্দ্রনাথ ঠাকুর"),
    ("Marathi", "फक्त मराठीत लिहा: महाराष्ट्र"),
    ("Gujarati", "ફક્ત ગુજરાતીમાં લખો: ગુજરાત"),
    ("Malayalam", "മലയാളത്തിൽ മാത്രം എഴുതുക: കേരളം"),
    ("Punjabi", "ਸਿਰਫ਼ ਪੰਜਾਬੀ ਵਿੱਚ ਲਿਖੋ: ਪੰਜਾਬ"),
]

for lang, prompt in tests:
    print(f"\n{lang}:")
    print(f"  Input: {prompt}")
    output = test_gen(prompt)
    
    # Check for English mixing
    english_chars = sum(1 for c in output if c.isalpha() and ord(c) < 128)
    total_alpha = sum(1 for c in output if c.isalpha())
    english_pct = (english_chars / total_alpha * 100) if total_alpha > 0 else 0
    
    print(f"  Output: {output[:120]}...")
    status = "✅" if english_pct < 30 else "⚠️" if english_pct < 50 else "❌"
    print(f"  English mixing: {english_pct:.1f}% {status}")
    print("-" * 80)

print("\n✅ Testing complete!")


## 📥 Step 9: Download & Deploy

### How to deploy on your PC:

1. **Download** `gurukul_lite_enhanced.zip` (click Files 📁 → Right-click → Download)
2. **Extract** the zip on your PC
3. **Backup** current adapter:
   ```bash
   cd C:\pc\Project
   rename adapters\gurukul_lite adapters\gurukul_lite_backup
   ```
4. **Copy** extracted folder to `adapters/gurukul_lite`
5. **Restart** server:
   ```bash
   python -m uvicorn src.api.main:app --host 127.0.0.1 --port 8117
   ```
6. **Test** accuracy:
   ```bash
   python test_accuracy_detailed.py
   ```

### Expected Results:
- ✅ Accuracy: 66.7% → 80-85%
- ✅ English mixing: Reduced significantly
- ✅ Better topic adherence
- ✅ Ready for adding more languages!
